# LoRA vs. Evolutionary-LoRA — OOD base arithmetic

**Hypothesis.** An *evolutionary* LoRA schedule — **dropout as a diversity pressure**,
**weight decay as a selection pressure** — forces genuine circuit formation, so the adapter
generalizes the arithmetic *algorithm* to unseen number bases instead of memorizing surface
patterns. We test it head-to-head against standard LoRA.

**Task.** base-`b` arithmetic (add / sub / mul on 2-digit numbers, all mod `b²`) for bases
7–16, chained into compositional **levels 1–5** (level N = N ops chained). Digits are
hex-style `0-9 A-F`.

**The split that makes it a test**
- **Train:** bases 7–14 (digits `0–D`), levels 1–5.
- **Val / Test:** levels **3–5 only** (the hard compositions), each split into
  *in-distribution* (bases 7–14, disjoint from train) and *OOD* (bases 15–16 — digits
  **`E`,`F` never seen in training**).
- **Val-OOD is the deciding metric:** only it measures genuine algorithm generalization;
  Val-ID can be won by memorization.

**Two conditions, identical data & seed**
- **standard** — fixed dropout 0.05, no weight-decay schedule; swept over LoRA ranks
  `{1,4,16,64,256}`, short horizon (plateaus fast).
- **evolutionary** — dropout `0.05→0.30→0.05` and weight decay `0.01→0.20→0.10` over
  training; single rank 256, long horizon.

Early stopping is **disabled** — we log the whole trajectory and compare each condition's
*best* Val-OOD.

---

## Pipeline (run top to bottom)

| # | cell | what it does |
|---|------|--------------|
| 1 | Dependencies + HF auth | install libs (Colab); log in via the `HF_TOKEN` Colab secret |
| 2 | Config | every knob: model, splits, LoRA, training, the evo schedule points |
| 3 | Paths | mount Drive; set `DATA_DIR` and per-model `RESULTS_DIR` |
| 4 | Serialization | CSV row → instructed prompt + answer string |
| 5 | Splits | build train / val / test pools (leakage-guarded) |
| 6 | Preview | print one example prompt per base — see exactly what the model gets |
| 7 | Tokenizer + masking | encode prompt+answer; **loss masked to the answer tokens only** |
| 8 | Model factory | frozen base + a fresh LoRA adapter per run |
| 9 | Schedules | `ParamScheduler` driving dropout & weight decay from training progress |
| 10 | Training loop + eval | manual loop: grad-accum, clipping, per-step schedule, generation eval |
| 11 | Baseline | zero-shot frozen model over all levels×bases; finds the ceiling C* |
| 12–13 | Quick tests | a few-step run of each condition to sanity-check the path |
| 14 | Full run | standard rank sweep, then the evolutionary run |
| 15 | Plots | Val-ID / Val-OOD trajectories + best-vs-rank (the deciding comparison) |
| 16 | Final test | held-out test sets, scored once |
| 17 | Save | adapters + results JSON + figures → `results/<model>/{baseline,standard,evo}/` |

**First run:** keep `SMOKE_TEST=True` (tiny splits/horizons, runs in ~a minute) to validate
end-to-end, then set it `False` for the real A100 run. See the **Notes** cell at the bottom
for tuning knobs and caveats.

In [ ]:
# --- Dependencies + HF auth (Colab only) ---------------------------------
# Skipped automatically when not running in Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q 'transformers>=4.51' 'peft>=0.13' 'datasets>=2.19' accelerate

# HF login for gated models (Llama, Gemma, some Mistral).
# Store your token in Colab: left sidebar -> key icon (Secrets) -> name it HF_TOKEN,
# paste the token, and toggle 'Notebook access' ON for this notebook.
if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    try:
        login(userdata.get('HF_TOKEN'))
        print('HF login OK')
    except Exception as e:
        print('HF login skipped (set the HF_TOKEN secret for gated models):', repr(e))

In [ ]:
# --- Config ---------------------------------------------------------------
import os, random
import numpy as np

SMOKE_TEST = True          # True = tiny sanity run (small splits, short horizons); False = full A100 run

# Pick ONE model (uncomment exactly one) -- run the suite one model at a time.
# Verify exact repo ids / base-vs-instruct against the HF Hub before a real run.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
# MODEL_NAME = 'Qwen/Qwen2.5-3B'
# MODEL_NAME = 'Qwen/Qwen3-4B'
# MODEL_NAME = 'meta-llama/Llama-3.2-1B'                  # gated
# MODEL_NAME = 'meta-llama/Llama-3.2-3B'                  # gated
# MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
# MODEL_NAME = 'microsoft/Phi-4-mini-instruct'
# MODEL_NAME = 'mistralai/Ministral-3B-Instruct-2410'    # verify id; may be gated
# MODEL_NAME = 'google/gemma-3-4b-pt'                     # gated (accept Gemma license)

WIDTH = 2                  # digits per number; MUST match the data generator's --max_digits

TRAIN_LEVELS = [1, 2, 3, 4, 5]    # training draws from all levels (bases 7-14)
EVAL_LEVELS  = [3, 4, 5]          # val/test use only these (harder compositions)
LOAD_LEVELS  = sorted(set(TRAIN_LEVELS) | set(EVAL_LEVELS))

# Fixed-count splits. SMOKE_TEST uses the tiny 'test' config to validate the pipeline.
SPLIT = 'test' if SMOKE_TEST else 'full'
SPLIT_CONFIGS = {
    'full': {'train': 50_000, 'val_id': 100, 'val_ood': 100, 'test_id': 2_000, 'test_ood': 2_000},
    'test': {'train': 200,    'val_id': 20,  'val_ood': 20,  'test_id': 50,    'test_ood': 50},
}

# LoRA
LORA_DROPOUT = 0.05                # standard/start dropout; MUST be > 0 so it stays mutable
LORA_TARGETS = 'all-linear'        # adapt ALL linear layers incl. MLP -- arithmetic lives in the MLPs
STD_RANKS = [4, 256] if SMOKE_TEST else [1, 4, 16, 64, 256]   # standard LoRA: full rank sweep (each plateaus early)
EVO_RANK  = 256                                               # evolutionary LoRA: single high rank
# lora_alpha tracks 2*rank (effective scale = 2) inside build_model

# Training (early stopping DISABLED -- we watch the whole trajectory)
LR           = 3e-4
STD_EPOCHS   = 2                        # standard plateaus fast -> short horizon for the sweep
EVO_EPOCHS   = 1 if SMOKE_TEST else 20  # evolutionary needs the long horizon
BATCH_SIZE   = 8
GRAD_ACCUM   = 2
GRAD_CLIP    = 1.0                 # max grad global-norm (guards vs exploding grads)
WEIGHT_DECAY = 0.0                 # standard-LoRA wd; evolutionary overrides via the schedule
EVAL_POINTS  = 8 if SMOKE_TEST else 120
EFF_RANK_EVERY       = 8           # baseline cadence: measure LoRA effective rank every Nth eval round (SVD is pricey)
EFF_RANK_DENSE_EVERY = 4           # denser cadence WHILE weight decay is rising -- where rank collapse happens
EFF_RANK_POWER_TOL   = 1e-3        # a singular component counts toward rank only if its power sigma^2 >= tol * sigma_max^2
LOG_EVERY    = 10

# Evolutionary schedule -- dropout & weight decay as functions of progress p in [0,1]
#   dropout: 0.05 until 10%, ramp to 0.20 by 20%, on to 0.30 peak at 33%, then taper to 0.05
#   wd:      0.01 until 33%, ramp up fast to 0.20 by 60%, then ease down to 0.10 by 100%
EVO_DROPOUT_POINTS = [(0.00, 0.05), (0.10, 0.05), (0.20, 0.20), (0.33, 0.30), (1.00, 0.05)]
EVO_WD_POINTS      = [(0.00, 0.01), (0.33, 0.01), (0.60, 0.20), (1.00, 0.10)]

SEED = 0

random.seed(SEED); np.random.seed(SEED)

In [ ]:
# --- Locate data + results dirs (Google Drive) ----------------------------
# Datasets at  DRIVE_DIR/base_datasets/level_*.csv
# Results at   DRIVE_DIR/results/<MODEL_SLUG>/{baseline,standard,evo}/
DRIVE_DIR = '/content/drive/MyDrive/sandbox_lora'   # your project folder on Drive -- EDIT to match

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = DRIVE_DIR
else:
    ROOT = '.'                                       # local fallback for the SMOKE_TEST run

DATA_DIR    = os.path.join(ROOT, 'base_datasets')
MODEL_SLUG  = MODEL_NAME.split('/')[-1]              # e.g. 'Qwen2.5-0.5B'
RESULTS_DIR = os.path.join(ROOT, 'results', MODEL_SLUG)
os.makedirs(RESULTS_DIR, exist_ok=True)

missing = [l for l in LOAD_LEVELS if not os.path.exists(os.path.join(DATA_DIR, f'level_{l}.csv'))]
assert not missing, f'Missing level CSVs {missing} under {DATA_DIR!r} - check DRIVE_DIR.'
print('data:', DATA_DIR, '| results:', RESULTS_DIR)

In [ ]:
# --- Row serialization (instructed prompt template) -----------------------
# A row -> (prompt, answer). The prompt sets the role, enumerates the base's
# digits, states the mod-100 closure, writes the left-nested expression with
# +, -, x symbols, and ends 'In base b, answer:' for the model to complete.
OPERAND_COLS = ['a', 'b', 'c', 'd', 'e', 'f']
_DIGITS = '0123456789ABCDEF'
_SYM = {'add': '+', 'sub': '-', 'mul': 'x'}

def op_cols(level):
    return ['task'] if level == 1 else [f'op{i}' for i in range(1, level + 1)]

def digit_list(base):
    return ', '.join(_DIGITS[i] for i in range(base))

def build_expression(ops, operands):
    """Left-nest with parentheses: ((a op1 b) op2 c) op3 d ..."""
    expr = f'{operands[0]} {_SYM[ops[0]]} {operands[1]}'
    for i in range(1, len(ops)):
        expr = f'({expr}) {_SYM[ops[i]]} {operands[i + 1]}'
    return expr

def make_prompt(base, ops, operands):
    digits = digit_list(base)
    expr   = build_expression(ops, operands)
    mod    = '1' + '0' * WIDTH        # 'mod 100' in base b == base**WIDTH
    return (
        f'You are a math expert solving a problem in base {base}.\n\n'
        f'In base {base}, the digits are: {digits}, '
        f'and 10 which equals {base} in decimal.\n\n'
        f'All numbers are in base {base}. '
        f'All results are mod {mod} (base {base}).\n\n'
        f'{expr}\n\n'
        f'In base {base}, answer:'
    )

def render(row, level):
    ops      = [row[c] for c in op_cols(level)]
    operands = [row[OPERAND_COLS[i]] for i in range(level + 1)]
    prompt   = make_prompt(int(row['base']), ops, operands)
    answer   = ' ' + row['answer']    # leading space -> clean ' 86' completion
    return prompt, answer

In [ ]:
# --- Load CSVs and build splits -------------------------------------------
# 1) Train: ALL base 7-14 rows from levels 1-5, randomly sampled to TRAIN size,
#    then REMOVED from the pool so val/test can never see a trained row.
# 2) Val/Test-ID:  bases 7-14, levels 3-5 only, drawn from the remaining rows.
# 3) Val/Test-OOD: bases 15-16, levels 3-5 only (never trained on).
# Leakage is guarded by PROMPT CONTENT (not object identity), so a trained
# problem's textual duplicate can never resurface in val/test.
import pandas as pd

cfg = SPLIT_CONFIGS[SPLIT]
rng = random.Random(SEED)

def load_records(levels):
    recs = []
    for level in levels:
        df = pd.read_csv(os.path.join(DATA_DIR, f'level_{level}.csv'), dtype=str)
        for _, r in df.iterrows():
            prompt, answer = render(r, level)
            recs.append({
                'base': int(r['base']), 'level': level,
                'prompt': prompt, 'answer': answer, 'answer_str': r['answer'],
            })
    return recs

all_recs = load_records(LOAD_LEVELS)
id_recs  = [r for r in all_recs if r['base'] <= 14]
ood_recs = [r for r in all_recs if r['base'] >= 15]
rng.shuffle(id_recs); rng.shuffle(ood_recs)

def take_stratified(pool, n, levels):
    """Draw ~n rows, balanced across `levels`, in (already shuffled) pool order."""
    per = {l: [r for r in pool if r['level'] == l] for l in levels}
    out, i = [], 0
    while len(out) < n and any(per[l] for l in levels):
        l = levels[i % len(levels)]
        if per[l]:
            out.append(per[l].pop())
        i += 1
    return out

# 1) training set from all levels in the ID pool
train = id_recs[:cfg['train']]
used  = {r['prompt'] for r in train}             # content keys -> dedup by problem, not by object

# 2) ID val/test from levels 3-5, excluding any row whose prompt was trained on
id_hard = [r for r in id_recs if r['level'] in EVAL_LEVELS and r['prompt'] not in used]
val_id  = take_stratified(id_hard, cfg['val_id'], EVAL_LEVELS)
used   |= {r['prompt'] for r in val_id}
test_id = take_stratified([r for r in id_hard if r['prompt'] not in used], cfg['test_id'], EVAL_LEVELS)

# 3) OOD val/test from levels 3-5
ood_hard = [r for r in ood_recs if r['level'] in EVAL_LEVELS]
val_ood  = take_stratified(ood_hard, cfg['val_ood'], EVAL_LEVELS)
used_o   = {r['prompt'] for r in val_ood}
test_ood = take_stratified([r for r in ood_hard if r['prompt'] not in used_o], cfg['test_ood'], EVAL_LEVELS)

for name, s in [('train', train), ('val_id', val_id), ('val_ood', val_ood),
                ('test_id', test_id), ('test_ood', test_ood)]:
    by_lvl = {l: sum(1 for r in s if r['level'] == l) for l in sorted({r['level'] for r in s})}
    print(f'  {name:>8s}: {len(s):>6,}   by level {by_lvl}')

In [ ]:
# --- Preview: what the model actually sees --------------------------------
# One rendered example per base (incl. OOD 15-16), straight from `render`.
# Note how the digit roster grows with the base and how E/F only appear for 15-16.
_pool = id_recs + ood_recs
for _b in [7, 10, 13, 15, 16]:
    _ex = next((x for x in _pool if x['base'] == _b), None)
    if _ex is None:
        continue
    _tag = '   <-- OOD (held out)' if _b >= 15 else ''
    print('=' * 72)
    print(f'  base {_b}   level {_ex["level"]}{_tag}')
    print('=' * 72)
    print(_ex['prompt'])
    print(f'  >>> answer (training target): {_ex["answer_str"]}\n')

In [ ]:
# --- Tokenizer, answer-only masking, encoded train set, step budget -------
import torch
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(rec):
    # BOS (if the tokenizer adds one) on the prompt to MATCH eval-time tokenization;
    # answer takes no special tokens so nothing sneaks into the middle of the sequence.
    p = tok(rec['prompt'], add_special_tokens=True)['input_ids']
    a = tok(rec['answer'], add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    return {'input_ids': p + a, 'labels': [-100] * len(p) + a}   # loss on answer + EOS only

train_enc = [encode(r) for r in train]

# --- Tokenizer sanity: does the ' answer' format tokenize cleanly here? ----
# The leading space in render() makes the answer's first token space-prefixed, so
# it tokenizes the SAME whether alone or glued onto the prompt -- and matches what
# the model must emit at inference (prompt ends 'answer:' with no trailing space).
# Confirm per model: (1) the answer fits the generation budget (WIDTH+1 new tokens),
# (2) the prompt|answer boundary is stable (concat == joint tokenization).
def _tok_report(rec):
    a_ids  = tok(rec['answer'], add_special_tokens=False)['input_ids']
    pieces = [tok.decode([t]) for t in a_ids]
    joined = tok(rec['prompt'], add_special_tokens=True)['input_ids'] + a_ids
    whole  = tok(rec['prompt'] + rec['answer'], add_special_tokens=True)['input_ids']
    fits, stable = len(a_ids) <= WIDTH + 1, joined == whole
    print(f"  base {rec['base']:>2}  ans {rec['answer_str']!r:>6} -> {len(a_ids)} tok "
          f"{pieces}   budget_ok={fits}  boundary_stable={stable}")
    return fits and stable

# Probe an ID example and an OOD one (bases 15-16 exercise the held-out E/F digits).
_probe = train[:1] + [x for x in (id_recs + ood_recs) if x['base'] >= 15][:1]
print(f'answer tokenization for {MODEL_NAME} (target must fit {WIDTH + 1} generated tokens):')
_clean = all(_tok_report(r) for r in _probe)
print('  -> all clean' if _clean
      else '  -> *** budget or boundary issue for THIS tokenizer; see flags above ***')

def collate(batch):
    maxlen = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    ids, labels, attn = [], [], []
    for b in batch:
        n = maxlen - len(b['input_ids'])
        ids.append(b['input_ids'] + [pad] * n)
        labels.append(b['labels'] + [-100] * n)          # padding ignored in loss
        attn.append([1] * len(b['input_ids']) + [0] * n)  # padding ignored in attention
    return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels),
            'attention_mask': torch.tensor(attn)}

# Optimizer-step budget (one .step() per BATCH_SIZE*GRAD_ACCUM samples).
# Horizon differs by condition, so train_lora derives its own total from `epochs`.
steps_per_epoch = max(len(train_enc) // (BATCH_SIZE * GRAD_ACCUM), 1)

def horizon(epochs):
    total = steps_per_epoch * epochs
    return total, max(total // EVAL_POINTS, 1)   # (total_steps, eval_every)

print(f'steps/epoch {steps_per_epoch}   '
      f'standard {steps_per_epoch * STD_EPOCHS} steps   '
      f'evolutionary {steps_per_epoch * EVO_EPOCHS} steps')

In [ ]:
# --- Model factory (a fresh frozen base + LoRA adapter per run) ------------
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
bf16_ok    = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
LOAD_DTYPE = torch.bfloat16 if bf16_ok else torch.float32

def build_model(rank=EVO_RANK):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=LOAD_DTYPE, trust_remote_code=True,
    )
    base.config.pad_token_id = tok.pad_token_id
    base.to(DEVICE)
    base.enable_input_require_grads()   # needed for gradient checkpointing + LoRA
    lora = LoraConfig(
        r=rank, lora_alpha=2 * rank, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, bias='none', task_type='CAUSAL_LM',
    )
    model = get_peft_model(base, lora)
    for p in model.parameters():        # keep trainable LoRA params in fp32 for AdamW stability
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()
    return model

In [ ]:
# --- Schedules: dropout & weight decay as functions of progress -----------
# The evolutionary method in one place:
#   DROPOUT = diversity pressure  -> high early, so the optimizer explores many
#       candidate circuits instead of collapsing straight into a surface memorizer.
#   WEIGHT DECAY = selection pressure -> ramped up later to erode high-norm
#       memorization circuits; cheaper low-norm 'logic' circuits survive, which
#       drives the adapter's effective rank down.
# Each schedule is a function  p in [0,1] -> value,  driven by p = step / total_steps.
import torch.nn as nn

def constant(value):
    return lambda p: value

def piecewise_linear(points):
    """Linear interpolation through (progress, value) control points; ends clamped."""
    pts = sorted(points)
    def f(p):
        p = min(max(p, 0.0), 1.0)
        if p <= pts[0][0]:  return pts[0][1]
        if p >= pts[-1][0]: return pts[-1][1]
        for (p0, v0), (p1, v1) in zip(pts, pts[1:]):
            if p0 <= p <= p1:
                t = (p - p0) / (p1 - p0) if p1 > p0 else 0.0
                return v0 + t * (v1 - v0)
        return pts[-1][1]
    return f

class ParamScheduler:
    """Set LoRA dropout and optimizer weight decay from progress p = step / total."""
    def __init__(self, total_steps, dropout_fn, wd_fn):
        self.total = max(int(total_steps), 1)
        self.dropout_fn, self.wd_fn = dropout_fn, wd_fn

    def values(self, step):
        p = min(step / self.total, 1.0)
        return float(self.dropout_fn(p)), float(self.wd_fn(p))

    def set(self, model, optimizer, step):
        do, wd = self.values(step)
        for m in model.modules():                       # mutate every real LoRA dropout module
            ld = getattr(m, 'lora_dropout', None)
            if isinstance(ld, nn.ModuleDict):
                for d in ld.values():
                    if isinstance(d, nn.Dropout):
                        d.p = do
        for g in optimizer.param_groups:                # decoupled AdamW weight decay
            g['weight_decay'] = wd
        return do, wd

def evo_scheduler(total_steps):
    return ParamScheduler(total_steps,
                          piecewise_linear(EVO_DROPOUT_POINTS),
                          piecewise_linear(EVO_WD_POINTS))

In [ ]:
# --- Training loop + evaluation -------------------------------------------
# A manual loop (not HF Trainer) because the evolutionary method mutates dropout
# and weight decay EVERY optimizer step via the scheduler -- which Trainer's built-in
# loop won't do. One function runs both conditions: evolutionary=False -> standard
# (no schedule); evolutionary=True -> attach the ParamScheduler. Also here: gradient
# accumulation, grad-norm clipping, and greedy generation exact-match eval.
import itertools, gc
from contextlib import nullcontext
from torch.utils.data import DataLoader

def release():
    """Reclaim GPU memory from tensors that just lost their last reference."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_mb():
    return torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0.0

def amp():
    return torch.autocast(device_type='cuda', dtype=torch.bfloat16) if bf16_ok else nullcontext()

@torch.no_grad()
def effective_rank(model, adapter='default', power_tol=None):
    """Mean entropy-based effective rank of each layer's LoRA delta-W = scaling * B @ A.

    Cheap: QR the thin factors and SVD only the r x r core, so the full out x in
    delta-W is never materialized (matters at rank 256 x all-linear). A singular
    component must clear a POWER floor to count toward the rank: keep sigma_i only
    where sigma_i^2 >= power_tol * sigma_max^2; weaker components are numerical
    noise and are dropped before the Roy-Vetterli effective-rank entropy. Returns
    the mean effective rank over all adapted layers (NaN if there are none)."""
    tol = EFF_RANK_POWER_TOL if power_tol is None else power_tol
    eranks = []
    for m in model.modules():
        A_dict, B_dict = getattr(m, 'lora_A', None), getattr(m, 'lora_B', None)
        if not (isinstance(A_dict, nn.ModuleDict) and adapter in A_dict):
            continue
        A = A_dict[adapter].weight.float()            # (r, in)
        B = B_dict[adapter].weight.float()            # (out, r)
        s = float(m.scaling[adapter])
        Rb = torch.linalg.qr(B,   mode='r').R         # (<=r, r)   thin-factor cores
        Ra = torch.linalg.qr(A.T, mode='r').R         # (<=r, r)
        sigma = torch.linalg.svdvals(s * (Rb @ Ra.T)) # same nonzero spectrum as delta-W
        pmax = (sigma ** 2).max()
        if pmax <= 0:                                 # delta-W == 0 (e.g. step 0: lora_B inits to 0)
            eranks.append(0.0)
            continue
        sigma = sigma[sigma ** 2 >= tol * pmax]       # power floor: drop noise components
        p = sigma / sigma.sum()                       # normalized spectrum
        eranks.append(float(torch.exp(-(p * p.log()).sum())))   # exp(spectral entropy)
    return sum(eranks) / len(eranks) if eranks else float('nan')

@torch.no_grad()
def evaluate(model, recs, bs=64):
    """Greedy-decode the answer and score exact string match."""
    if not recs:
        return 0.0
    model.eval()
    correct = 0
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        enc = tok([r['prompt'] for r in chunk], return_tensors='pt', padding=True).to(DEVICE)
        with amp():
            out = model.generate(**enc, max_new_tokens=WIDTH + 1, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        gen = out[:, enc['input_ids'].shape[1]:]
        for r, g in zip(chunk, gen):
            correct += (tok.decode(g, skip_special_tokens=True).strip() == r['answer_str'])
    model.train()
    return correct / len(recs)

def train_lora(evolutionary=False, rank=EVO_RANK, epochs=EVO_EPOCHS, steps=None, tag='standard'):
    """Train one LoRA adapter. `steps` overrides the epoch-derived horizon (for quick tests)."""
    torch.manual_seed(SEED)             # same seed across runs -> controlled comparison
    total_steps, eval_every = horizon(epochs)
    if steps is not None:
        total_steps = steps
        eval_every  = max(steps // max(EVAL_POINTS, 1), 1)
    model = build_model(rank)
    if not SMOKE_TEST:
        model.gradient_checkpointing_enable()
    model.train()
    tok.padding_side = 'left'           # left-pad for batched generation during eval

    params = [p for p in model.parameters() if p.requires_grad]
    opt    = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = evo_scheduler(total_steps) if evolutionary else None

    def wd_rising(step):
        """True while the evo weight-decay schedule is in an increasing segment at `step`."""
        if sched is None:
            return False
        p = min(step / sched.total, 1.0)
        return sched.wd_fn(min(p + 1e-3, 1.0)) > sched.wd_fn(p) + 1e-12

    loader  = DataLoader(train_enc, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate, drop_last=True)
    batches = itertools.cycle(loader)
    hist = {'step': [], 'val_id': [], 'val_ood': [], 'dropout': [], 'wd': [],
            'loss': [], 'grad_norm': [], 'eff_rank': []}
    n_eval = 0                          # eval-round counter -> gate the sparse SVD measurement

    def record(step):
        nonlocal n_eval
        vid, vood = evaluate(model, val_id), evaluate(model, val_ood)
        do, wd = (sched.values(step) if sched else (LORA_DROPOUT, WEIGHT_DECAY))
        hist['step'].append(step); hist['val_id'].append(vid); hist['val_ood'].append(vood)
        hist['dropout'].append(do); hist['wd'].append(wd)
        er_msg = ''
        every = EFF_RANK_DENSE_EVERY if wd_rising(step) else EFF_RANK_EVERY  # densify on the wd ramp
        if n_eval % every == 0:                       # SVD effective rank, sparsely
            er = effective_rank(model)
            hist['eff_rank'].append((step, er))
            er_msg = f'   eff_rank {er:5.1f}'
        n_eval += 1
        print(f'[{tag}] {step:>6}/{total_steps}  val_id {vid:5.1%}  val_ood {vood:5.1%}'
              f'   do {do:.3f}  wd {wd:.3f}{er_msg}')

    record(0)
    for step in range(1, total_steps + 1):
        if sched:
            sched.set(model, opt, step)   # update dropout (forward) + weight decay (update) for this step
        opt.zero_grad(set_to_none=True)
        running = 0.0
        for _ in range(GRAD_ACCUM):
            batch = {k: v.to(DEVICE) for k, v in next(batches).items()}
            with amp():
                loss = model(**batch).loss / GRAD_ACCUM
            loss.backward()
            running += loss.item()
        gnorm = torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)   # clip global norm; guard vs exploding grads
        if torch.isfinite(gnorm):
            opt.step()
        else:
            print(f'[{tag}] step {step}: non-finite grad norm; update skipped')
        if step % LOG_EVERY == 0:
            hist['loss'].append((step, running))
            hist['grad_norm'].append((step, float(gnorm)))
        if step % eval_every == 0 or step == total_steps:
            record(step)

    # Free what the caller does not need: optimizer (Adam moments) + gradient buffers.
    model.zero_grad(set_to_none=True)
    del opt, loader, batches
    return hist, model

In [ ]:
# --- Baseline (zero-shot, frozen model): all levels x all bases -----------
# Sample BASE_N rows per (level, base) over levels 1-5, bases 7-16, score the
# UN-finetuned model, graph accuracy by level and by base, and save to
# RESULTS_DIR/baseline/{baseline.json, baseline.png}.
#   full: 50 x 5 levels x 10 bases = 2500 rows  (500/level, 250/base)
import gc, json, math
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

ZS_MODELS   = [MODEL_NAME]
# Compare base vs instruct in one pass, e.g.:
# ZS_MODELS = ['Qwen/Qwen2.5-3B', 'Qwen/Qwen2.5-3B-Instruct']
BASE_LEVELS = [1, 2, 3, 4, 5]
BASE_BASES  = list(range(7, 17))          # 7..16
BASE_N      = 5 if SMOKE_TEST else 50     # samples per (level, base)

# Balanced probe set: BASE_N rows per (level, base).
base_recs = []
for level in BASE_LEVELS:
    df = pd.read_csv(os.path.join(DATA_DIR, f'level_{level}.csv'), dtype=str)
    cap = {b: 0 for b in BASE_BASES}
    for _, r in df.sample(frac=1, random_state=SEED).iterrows():
        b = int(r['base'])
        if cap.get(b, BASE_N) >= BASE_N:
            continue
        cap[b] += 1
        prompt, _ = render(r, level)
        base_recs.append({'base': b, 'level': level, 'prompt': prompt, 'answer_str': r['answer']})
        if all(v >= BASE_N for v in cap.values()):
            break
print(f'baseline probe set: {len(base_recs)} rows '
      f'({BASE_N}/level/base x {len(BASE_LEVELS)} levels x {len(BASE_BASES)} bases)')

@torch.no_grad()
def zs_predict(model, toker, recs, bs=64):
    if toker.pad_token is None:
        toker.pad_token = toker.eos_token
    toker.padding_side = 'left'
    preds = []
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        enc = toker([r['prompt'] for r in chunk], return_tensors='pt', padding=True).to(DEVICE)
        with amp():
            out = model.generate(**enc, max_new_tokens=WIDTH + 1, do_sample=False,
                                 pad_token_id=toker.pad_token_id)
        gen = out[:, enc['input_ids'].shape[1]:]
        preds += [toker.decode(g, skip_special_tokens=True).strip() for g in gen]
    return preds

def _acc(rows):
    return sum(r['correct'] for r in rows) / len(rows) if rows else float('nan')

def _json_safe(o):
    """Recursively replace non-finite floats (NaN/inf) with None so the dump is valid JSON."""
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, list):
        return [_json_safe(v) for v in o]
    return o

bdir = os.path.join(RESULTS_DIR, 'baseline'); os.makedirs(bdir, exist_ok=True)
baseline = {}   # results per model, also saved to disk
for zs_name in ZS_MODELS:
    print(f'\n=== baseline (zero-shot): {zs_name} ===')
    zs_tok   = AutoTokenizer.from_pretrained(zs_name, trust_remote_code=True)
    zs_model = AutoModelForCausalLM.from_pretrained(
        zs_name, torch_dtype=LOAD_DTYPE, trust_remote_code=True).to(DEVICE)
    zs_model.config.pad_token_id = (zs_tok.pad_token_id if zs_tok.pad_token_id is not None
                                    else zs_tok.eos_token_id)   # 0 is a valid id -> don't use `or`
    zs_model.eval()

    for r, p in zip(base_recs, zs_predict(zs_model, zs_tok, base_recs)):
        r['correct'] = int(p == r['answer_str'])

    by_level = {lv: _acc([r for r in base_recs if r['level'] == lv]) for lv in BASE_LEVELS}
    by_base  = {b: _acc([r for r in base_recs if r['base'] == b]) for b in BASE_BASES}
    grid = {lv: {b: _acc([r for r in base_recs if r['level'] == lv and r['base'] == b])
                 for b in BASE_BASES} for lv in BASE_LEVELS}
    C_star = next((lv for lv in BASE_LEVELS if by_level[lv] < 0.05), None)
    baseline[zs_name] = {'n_per_cell': BASE_N, 'by_level': by_level,
                         'by_base': by_base, 'grid': grid, 'C_star': C_star}
    print('by level:', {lv: f'{a:.0%}' for lv, a in by_level.items()})
    print('by base :', {b: f'{a:.0%}' for b, a in by_base.items()})
    print('compositional ceiling C* =', C_star, '(first level under 5%)')

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    ax[0].bar([str(l) for l in BASE_LEVELS], [by_level[l] for l in BASE_LEVELS], color='tab:purple')
    ax[0].set_title('baseline accuracy by level'); ax[0].set_xlabel('level')
    ax[0].set_ylabel('exact-match accuracy')
    ax[1].bar([str(b) for b in BASE_BASES], [by_base[b] for b in BASE_BASES],
              color=['tab:blue' if b <= 14 else 'tab:orange' for b in BASE_BASES])
    ax[1].set_title('baseline accuracy by base  (orange = OOD 15-16)'); ax[1].set_xlabel('base')
    for a in ax:
        a.set_ylim(0, 1.02); a.grid(alpha=0.3, axis='y')
    fig.suptitle(f'Zero-shot baseline: {zs_name.split("/")[-1]}')
    plt.tight_layout(); plt.show()

    # persist (one figure + json per model; suffix when comparing several)
    suffix = '' if len(ZS_MODELS) == 1 else '_' + zs_name.split('/')[-1]
    fig.savefig(os.path.join(bdir, f'baseline{suffix}.png'), bbox_inches='tight')
    json.dump(_json_safe({'model': zs_name, **baseline[zs_name]}),
              open(os.path.join(bdir, f'baseline{suffix}.json'), 'w'), indent=1)

    del zs_model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('saved baseline ->', bdir)

In [ ]:
# --- TEST: standard LoRA (a few steps, throwaway) -------------------------
# Quick end-to-end check of the standard path: builds, trains a handful of
# steps, evaluates, and frees. Does not touch the canonical run variables.
_h, _m = train_lora(evolutionary=False, rank=4, steps=20, tag='test-std')
print('\nval_ood over the test run:',
      [f'{v:.0%}' for v in _h['val_ood']])
del _m; release()
print(f'[mem] after test-std: {gpu_mb():.0f} MB on GPU')

In [ ]:
# --- TEST: evolutionary LoRA (a few steps, throwaway) ---------------------
# Sanity-checks the evo path AND that the dropout/weight-decay schedule fires.
# `steps` is small so progress sweeps 0->1 quickly and you see the curve move.
_h, _m = train_lora(evolutionary=True, rank=16, steps=40, tag='test-evo')
print('\nschedule actually applied (step -> dropout, wd):')
for _s, _do, _wd in zip(_h['step'], _h['dropout'], _h['wd']):
    print(f'  step {_s:>3}:  dropout {_do:.3f}   wd {_wd:.3f}')
del _m; release()
print(f'[mem] after test-evo: {gpu_mb():.0f} MB on GPU')

In [ ]:
# --- FULL RUN: standard LoRA rank sweep (short), then evolutionary LoRA ----
# (release() and gpu_mb() are defined in the training-loop cell.)

# Standard LoRA across all ranks, STD_EPOCHS each; keep only the best-Val-OOD model.
std_hists = {}
std_model, std_best_rank, std_best_ood = None, None, -1.0
for r in STD_RANKS:
    h, m = train_lora(evolutionary=False, rank=r, epochs=STD_EPOCHS, tag=f'std-r{r}')
    std_hists[r] = h
    peak = max(h['val_ood'])
    if peak > std_best_ood:
        old = std_model                          # the previous best (or None)
        std_model, std_best_rank, std_best_ood = m, r, peak
        del old                                  # release it now
    m = None                                     # drop the loop handle; a non-best model now has 0 refs
    release()                                    # -> actually frees it on the GPU
    print(f'  [mem] after std-r{r}: {gpu_mb():.0f} MB on GPU')
print(f'\nbest standard: rank {std_best_rank}  (Val-OOD {std_best_ood:.1%})')

# Park the kept standard model on CPU so it does not occupy GPU during the long evo run.
if std_model is not None and torch.cuda.is_available():
    std_model.to('cpu'); release()
    print(f'  [mem] standard parked on CPU: {gpu_mb():.0f} MB on GPU')

# Evolutionary LoRA: single high rank, EVO_EPOCHS (long horizon).
evo_hist, evo_model = train_lora(evolutionary=True, rank=EVO_RANK, epochs=EVO_EPOCHS,
                                 tag=f'evo-r{EVO_RANK}')
release()
print(f'  [mem] after evo: {gpu_mb():.0f} MB on GPU')

In [ ]:
# --- Plots: the deciding comparison ---------------------------------------
import matplotlib.pyplot as plt

ranks = sorted(std_hists)
blues = plt.cm.Blues(np.linspace(0.4, 0.95, len(ranks)))
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))

# (0) Val-ID trajectories (in-distribution; should climb high)
for c, r in zip(blues, ranks):
    ax[0].plot(std_hists[r]['step'], std_hists[r]['val_id'], '-', color=c, label=f'std r={r}')
ax[0].plot(evo_hist['step'], evo_hist['val_id'], '-o', ms=3, color='tab:red', label=f'evo r={EVO_RANK}')
ax[0].set_title('Val-ID  (bases 7-14, levels 3-5)')
ax[0].set_xlabel('optimizer step'); ax[0].set_ylabel('exact-match accuracy')

# (1) Val-OOD trajectories  <- the deciding metric
for c, r in zip(blues, ranks):
    ax[1].plot(std_hists[r]['step'], std_hists[r]['val_ood'], '-', color=c, label=f'std r={r}')
ax[1].plot(evo_hist['step'], evo_hist['val_ood'], '-o', ms=3, color='tab:red', label=f'evo r={EVO_RANK}')
ax[1].set_title('Val-OOD  (bases 15-16, levels 3-5)  <- the metric')
ax[1].set_xlabel('optimizer step')

# (2) best Val-OOD vs rank: standard sweep vs evolutionary reference line
std_best = [max(std_hists[r]['val_ood']) for r in ranks]
ax[2].plot(ranks, std_best, '-o', color='tab:blue', label='standard (best)')
ax[2].axhline(max(evo_hist['val_ood']), color='tab:red', ls='--', label=f'evo r={EVO_RANK}')
ax[2].set_xscale('log', base=2); ax[2].set_xticks(ranks); ax[2].set_xticklabels(ranks)
ax[2].set_title('best Val-OOD vs LoRA rank'); ax[2].set_xlabel('rank')

for a in ax:
    a.set_ylim(-0.02, 1.02); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

# save the comparison figure into both run-type folders
for sub in ('standard', 'evo'):
    d = os.path.join(RESULTS_DIR, sub); os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, 'comparison.png'), bbox_inches='tight')

# numeric summary: best Val-ID, best Val-OOD, generalization gap
print('best-over-run   Val-ID  /  Val-OOD  /  gap (ID-OOD)')
for r in ranks:
    vi, vo = max(std_hists[r]['val_id']), max(std_hists[r]['val_ood'])
    print(f'  std r{r:<4} {vi:6.1%}  / {vo:7.1%}  / {vi - vo:+.1%}')
vi, vo = max(evo_hist['val_id']), max(evo_hist['val_ood'])
print(f'  evo r{EVO_RANK:<3} {vi:6.1%}  / {vo:7.1%}  / {vi - vo:+.1%}')

In [ ]:
# --- Final test-set eval (run ONCE, after all dev decisions) --------------
# Held-out test sets (levels 3-5). Touch these only at the very end.
if torch.cuda.is_available():
    std_model.to(DEVICE)   # bring the parked standard model back onto the GPU to score

test_results = {}
print(f'{"model":>18}   test_id   test_ood   gap')
for model, name in [(std_model, f'standard r{std_best_rank}'),
                    (evo_model, f'evolutionary r{EVO_RANK}')]:
    tid, tood = evaluate(model, test_id), evaluate(model, test_ood)
    test_results[name] = {'test_id': tid, 'test_ood': tood, 'gap': tid - tood}
    print(f'{name:>18}   {tid:6.1%}   {tood:7.1%}   {tid - tood:+.1%}')

In [ ]:
# --- Save: adapters + results, per run type, at the end -------------------
# results/<MODEL_SLUG>/standard/  -> best-standard adapter + standard.json + comparison.png
# results/<MODEL_SLUG>/evo/        -> evolutionary adapter + evo.json + comparison.png
# (baseline/ was written by the baseline cell.)  save_pretrained writes ONLY the
# LoRA adapter (adapter_model.safetensors + adapter_config.json), a few MB.
# _json_safe (from the baseline cell) scrubs any non-finite grad-norm/eff-rank
# entries to null so the dumps stay valid JSON.
import json

cfg = {'model': MODEL_NAME, 'WIDTH': WIDTH, 'SPLIT': SPLIT, 'split_sizes': SPLIT_CONFIGS[SPLIT],
       'LORA_DROPOUT': LORA_DROPOUT, 'LORA_TARGETS': LORA_TARGETS, 'LR': LR,
       'STD_RANKS': STD_RANKS, 'EVO_RANK': EVO_RANK, 'STD_EPOCHS': STD_EPOCHS, 'EVO_EPOCHS': EVO_EPOCHS,
       'BATCH_SIZE': BATCH_SIZE, 'GRAD_ACCUM': GRAD_ACCUM, 'EVAL_POINTS': EVAL_POINTS,
       'EVO_DROPOUT_POINTS': EVO_DROPOUT_POINTS, 'EVO_WD_POINTS': EVO_WD_POINTS}
tr = globals().get('test_results', {})

sdir = os.path.join(RESULTS_DIR, 'standard'); os.makedirs(sdir, exist_ok=True)
std_model.save_pretrained(sdir)              # adapter files into standard/
json.dump(_json_safe({'config': cfg, 'std_best_rank': std_best_rank, 'std_hists': std_hists, 'test': tr}),
          open(os.path.join(sdir, 'standard.json'), 'w'), indent=1)
print('saved standard ->', sdir)

edir = os.path.join(RESULTS_DIR, 'evo'); os.makedirs(edir, exist_ok=True)
evo_model.save_pretrained(edir)              # adapter files into evo/
json.dump(_json_safe({'config': cfg, 'evo_rank': EVO_RANK, 'evo_hist': evo_hist, 'test': tr}),
          open(os.path.join(edir, 'evo.json'), 'w'), indent=1)
print('saved evo ->', edir)

# Reload later to replot:  run = json.load(open(os.path.join(sdir, 'standard.json')))
# Reload an adapter:
#   from peft import PeftModel
#   base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=LOAD_DTYPE)
#   model = PeftModel.from_pretrained(base, sdir)

## Notes and knobs

- **Run order.** Pick one `MODEL_NAME`, keep `SMOKE_TEST=True` for a fast end-to-end check, then set `SMOKE_TEST=False` and Run all. Standard LoRA sweeps `STD_RANKS`; evolutionary runs once at `EVO_RANK=256`. The plot overlays all of them on Val-OOD.
- **Split horizons.** `STD_EPOCHS=2` (standard plateaus fast, so the sweep stays cheap) vs. `EVO_EPOCHS=20` (evolutionary needs the long horizon). Each `train_lora` derives its own `total_steps` and eval cadence from its `epochs`, so both still land `EVAL_POINTS` eval rounds. *(Caveat: evo therefore trains ~10x longer than each standard run — to attribute a win to the schedule rather than the horizon, add a standard r=256 control at `EVO_EPOCHS`.)*
- **Rank design.** Sweep `{1,4,16,64,256}` for standard and keep the best-Val-OOD model; evolutionary pinned to r=256. `lora_alpha` tracks `2*rank` automatically.
- **Effective rank.** `effective_rank` measures the entropy-based rank of each layer's LoRA ΔW (factored SVD, never materializing ΔW). It runs every `EFF_RANK_EVERY` eval rounds, densifying to `EFF_RANK_DENSE_EVERY` while weight decay is rising. A singular component counts only if its power σ² clears `EFF_RANK_POWER_TOL`·σ²_max.
- **Gated models** (Llama, Gemma, some Mistral): request access on the Hub, then run `from huggingface_hub import login; login()` once (see the deps cell) before the model cell.
- **Memory.** Assumes a single **A100** with full **bf16** weights — 3-4B + LoRA fits comfortably. If you OOM, lower `BATCH_SIZE` and raise `GRAD_ACCUM` to keep the effective batch.
- **The schedule is data.** `EVO_DROPOUT_POINTS` / `EVO_WD_POINTS` are `(progress, value)` control points (progress = fraction of total steps). Edit to reshape; `constant(x)` gives a flat control. Default: dropout holds 0.05 to 10%, ramps to 0.20 by 20% and peaks 0.30 at 33%, then tapers back to 0.05 by 100%; weight decay holds 0.01 to 33%, ramps to 0.20 by 60%, then eases to 0.10 by 100%.
- **Dropout must stay > 0.** `LORA_DROPOUT=0.05` so PEFT creates real `nn.Dropout` modules the scheduler can mutate. Set it to 0 and PEFT inserts `nn.Identity`, so the schedule silently does nothing.
- **Early stopping is disabled** by design — the in-dist cutoff is just a reference; we compare *best* Val-OOD over the whole trajectory.
- **Tokenization caveat:** these tokenizers already have tokens for `E`/`F`, so Val-OOD measures whether the model can *use* pretrained symbol embeddings it never saw fine-tuned — not a truly unseen token.